## Data Loading and Cleaning for Experiment

In [1]:
import math
import numpy as np
import pandas as pd
import os
from dotenv import load_dotenv
from huggingface_hub import login
import gc
import torch
from tqdm import tqdm
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM

In [2]:
# get credentials from .env file and login to hugging face so we can run models
env_path = os.path.join( os.path.expanduser('~'), 'Documents', '.env')
load_dotenv(env_path)
hf_token = os.environ.get('HUGGINGFACE_API_TOKEN')
login(hf_token)

In [3]:
# ensure cache is empty so we can run
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'
torch.cuda.empty_cache()

In [4]:
# grab data from path
df_full = pd.read_csv('./data/all_prompts.csv', index_col = 0)
df_full.head()

,text,source,label
0,"Federal law supersedes state law, and cannabis...",Bloom-7B,1
1,Miles feels restless after working all day. He...,Bloom-7B,1
2,So first of I am danish. That means that I fol...,Bloom-7B,1
3,In this paper we present a novel rule-based ap...,Bloom-7B,1
4,"Most social progressives, love democracy, and ...",Bloom-7B,1


In [5]:
# ensure data loaded correctly 
df_full.columns

Index(['text', 'source', 'label'], dtype='object')

In [6]:
# get distribution of labels
# note: human is 0, bot is 1
df_full['label'].value_counts()

label
1    508061
0    414523
Name: count, dtype: int64

In [7]:
# subsample since the number of rows is extremely large
SAMPLE_SIZE = 10000
RANDOM_SEED = 213

# note we equally; we are assuming that it is equally likely that text is bot generated or human (may not actually be the case)
df = df_full.groupby('label', group_keys = False).apply(lambda x: x.sample(min(len(x), SAMPLE_SIZE // 2), random_state = RANDOM_SEED)).reset_index(drop = True)

C:\Users\Chris\AppData\Local\Temp\ipykernel_27336\2004147289.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df_full.groupby('label', group_keys = False).apply(lambda x: x.sample(min(len(x), SAMPLE_SIZE // 2), random_state = RANDOM_SEED)).reset_index(drop = True)


In [8]:
print(f"Shape: {df.shape}")
print('\n')
print(f"Distribution: {df['label'].value_counts()}")

Shape: (10000, 3)


Distribution: label
0    5000
1    5000
Name: count, dtype: int64


In [9]:
# add structural columns to mirror original experiment and allow it to reference choices
df['choices'] = [['Human-written', 'AI-generated']] * df.shape[0]
df['subject'] = 'AI_text_detection'
df['answerKey'] = df['label'].map({0: 'A', 1:'B'}) # remember: human = 0, bot = 1; this means human = A, bot = B

In [10]:
# define helper function for later; enable changing the max_length layer 
def get_dynamic_max_length(text):
    char_len = len(text)
    if char_len < 500: # short texts: tweets, comments, sentences
        return 256
    elif char_len < 1500: # medium texts: paragraphs, social media posts
        return 512
    elif char_len < 3000: # long texts: articles, blog posts
        return 768
    else: # very long texts: essays, academic papers
        return 1024

In [11]:
# see how long it will take per
df['max_length_used'] = df['text'].apply(get_dynamic_max_length)
df['max_length_used'].value_counts()

max_length_used
1024    3101
256     2683
512     2286
768     1930
Name: count, dtype: int64

In [12]:
df.head()

,text,source,label,choices,subject,answerKey,max_length_used
0,"After the American Revolutionary War, the numb...",Human,0,"[Human-written, AI-generated]",AI_text_detection,A,512
1,Apple Company: Problems and Solutions Essay Ex...,Human,0,"[Human-written, AI-generated]",AI_text_detection,A,1024
2,I don't like the physical location of the Fies...,Human,0,"[Human-written, AI-generated]",AI_text_detection,A,768
3,Post-Traumatic Stress Disorder in Veterans Res...,Human,0,"[Human-written, AI-generated]",AI_text_detection,A,1024
4,Assessment Informed Instruction Essay\n\nAs I ...,Human,0,"[Human-written, AI-generated]",AI_text_detection,A,1024


## Initial Generator Policy Functions

In [14]:
# ensures underscores are removed so tokens are reliably separated by strings for the subject (in this case it is always "AI_text_detection")
def format_subject(subject):
    return " ".join(subject.split("_"))

In [15]:
# # builds generator prompt
# # get correct = True -> prompt asks for the corect label (human or AI)
# # get correct = False -> prompt asks for the incorrect label

# def build_generator_prompt(subject, target_question, target_choices, get_correct):

#     prompt = "The following are texts for {}. Determine whether each text is human-written or AI-generated.\n\n".format(
#         format_subject(subject)
#     )
#     prompt += f"{target_question}"
#     for i, c in enumerate(target_choices):
#         prompt += "\n{}. {}".format(chr(65 + i), c)

#     if get_correct:
#         prompt += "\nAnswer:"
#     else:
#         prompt += "\nIncorrect Answer:"
#     return prompt

In [16]:
# changing generator prompt to reflect changes in discriminator prompt
def build_generator_prompt(subject, target_text, target_choices, get_correct):
    """
    Direct classification prompt for generator.
    get_correct=True  → asking for the correct label
    get_correct=False → asking for the incorrect label
    """
    if get_correct:
        prompt = (
            f"You are an expert at detecting AI-generated text.\n"
            f"Identify the correct label for the following text.\n"
            f"Output ONLY 'A' or 'B'.\n\n"
            f"Text: {target_text}\n\n"
            f"A. Human-written\n"
            f"B. AI-generated\n\n"
            f"Answer:"
        )
    else:
        prompt = (
            f"You are an expert at detecting AI-generated text.\n"
            f"Identify the incorrect label for the following text.\n"
            f"Output ONLY 'A' or 'B'.\n\n"
            f"Text: {target_text}\n\n"
            f"A. Human-written\n"
            f"B. AI-generated\n\n"
            f"Incorrect Answer:"
        )
    return prompt

In [17]:
# extracts next-token logits for each choice letter and performs softmax
# returns dict: {A:p_A, B:p_B}
def get_generator_answer_probs(model, tokenizer, prompt_text, choices_list, max_length = 512):
    input_ids = tokenizer(
        prompt_text, 
        return_tensors = "pt",
        truncation = True,
        max_length = max_length
    ).input_ids.to(model.device)

    with torch.no_grad():
        logits = model(input_ids=input_ids).logits[0, -1]

    choices = [chr(65 + i) for i in range(len(choices_list))]
    choice_logits = []
    for letter in choices:
        token_id = tokenizer(letter, return_tensors="pt").input_ids[0, -1].item()
        choice_logits.append(logits[token_id].item())

    choice_logits = torch.tensor(choice_logits, device=model.device).float()
    probs = torch.nn.functional.softmax(choice_logits, dim=0).detach().cpu().numpy()

    return {choice: float(prob) for choice, prob in zip(choices, probs)}

In [18]:
# gets the probabilities for a given prompt
def generator_probs(subject, text, choices_list, get_correct, model, tokenizer, max_length = 512):
    prompt = build_generator_prompt(subject, text, choices_list, get_correct)
    probs = get_generator_answer_probs(model, tokenizer, prompt, choices_list, max_length)
    return probs

In [19]:
# returns dictionary with initial generator probabilities for the prompts
def get_initial_generator_probs(row, model, tokenizer):
    max_length = get_dynamic_max_length(row["text"])
    gen_init = {"correct": {}, "incorrect": {}}
    x = row["text"]
    y_list = row["choices"]
    subject = row["subject"]
    for v in [True, False]:
        choice_probs = generator_probs(subject, x, y_list, v, model, tokenizer, max_length)
        key = "correct" if v else "incorrect"
        for letter, prob in choice_probs.items():
            gen_init[key][letter] = prob
    return gen_init

## Initial Discriminator Policies

In [22]:
# # builds the prompt as closely to the original experiment
# def build_discriminator_prompt(subject, text, proposed_label):
#     prompt = (
#         f"You are an expert evaluator of questions about {format_subject(subject)}.\n"
#         f"Determine if the proposed label is correct. Output ONLY 'A' or 'B'.\n"
#         f"Text: {text}\n"
#         f"Proposed Label: {proposed_label}\n\n"
#         f"Is this label correct? Respond ONLY with:\n"
#         f"A. Correct\n"
#         f"B. Incorrect\n\n"
#         f"Answer:"
#     )
#     return prompt

In [25]:
# the above prompt yielded extremely imbalanced results 
#  rerunninng with a different prompt
def build_discriminator_prompt(subject, text):
    prompt = (
        f"You are an expert at detecting AI-generated text.\n"
        f"Classify the following text as either human-written or AI-generated.\n"
        f"Output ONLY 'A' or 'B'.\n\n"
        f"Text: {text}\n\n"
        f"A. Human-written\n"
        f"B. AI-generated\n\n"
        f"Answer:"
    )
    return prompt

In [26]:
# extracts logits for discriminator
def get_discriminator_probs(model, tokenizer, prompt_text, choices_list, max_length = 512):
    input_ids = tokenizer(
        prompt_text, 
        return_tensors = "pt",
        truncation = True,
        max_length = max_length
    ).input_ids.to(model.device)
    with torch.no_grad():
        logits = model(input_ids=input_ids).logits[0, -1]

    choice_logits = torch.tensor(
        [
            logits[tokenizer("A").input_ids[-1]].item(),
            logits[tokenizer("B").input_ids[-1]].item(),
        ]
    ).float()

    probs = torch.nn.functional.softmax(choice_logits, dim=0).detach().cpu().numpy()
    disc_dict = {"A": "correct", "B": "incorrect"}
    choices = [chr(65 + i) for i in range(len(choices_list))]

    return {disc_dict[choice]: float(prob) for choice, prob in zip(choices, probs)}

In [27]:
# # iterates over each candidate label and evaluates all possible answers
# # returns dictionary
# # disc_init['A'] = is 'Human-written' the right label
# # disc_init['B'] = is 'AI-generated' the right label
# def evaluate_answer_correctness(row, model, tokenizer):
#     max_length = get_dynamic_max_length(row["text"])
#     subject = row["subject"]
#     text = row["text"]
#     choices = row["choices"]
    
#     results = {} # ['Human-written', 'AI-generated']
#     disc_dict_answer = {i: chr(65 + i) for i in range(len(choices))}
#     for idx, proposed_label in enumerate(choices):
#         prompt = build_discriminator_prompt(subject, text, proposed_label)
#         probs = get_discriminator_probs(model, tokenizer, prompt, choices, max_length)
#         results[disc_dict_answer[idx]] = probs
#     return results

In [28]:
# above function worked with old prompt
# function now changed to reflect structural changes to new prompt
def evaluate_answer_correctness(row, model, tokenizer):
    subject = row["subject"]
    text = row["text"]
    choices = row["choices"]   # ['Human-written', 'AI-generated']

    prompt = build_discriminator_prompt(subject = subject, text = text)
    
    input_ids = tokenizer(
        prompt, return_tensors="pt",
        truncation=True,
        max_length=get_dynamic_max_length(text)
    ).input_ids.to(model.device)

    with torch.no_grad():
        logits = model(input_ids=input_ids).logits[0, -1]

    token_a = tokenizer("A").input_ids[-1]
    token_b = tokenizer("B").input_ids[-1]

    p_a, p_b = torch.softmax(
        torch.tensor([logits[token_a], logits[token_b]]), dim=0
    ).tolist()

    # Match original dict structure: {choice: {correct: p, incorrect: 1-p}}
    # A's "correct" prob = p_a, B's "correct" prob = p_b
    return {
        "A": {"correct": p_a, "incorrect": 1 - p_a},
        "B": {"correct": p_b, "incorrect": 1 - p_b}
    }

In [29]:
def get_initial_discriminator_probs(row, model, tokenizer):
    return evaluate_answer_correctness(row, model, tokenizer)

## Equilibrium Search

In [31]:
# method = 'generator': pick argmax_y pi_G(correct|y)
# method = 'discriminator': pick argmax_y pi_D(correct|y)
# note this function is rewritten but functionally the same as the one in the original 
def pick_answer(gen, disc, candidates, method = "generator"):
    if method == "generator":
        return max(candidates, key = lambda y: gen["correct"][y])
    else:
        return max(candidates, key = lambda y: disc[y]["correct"])

In [32]:
# numerically stable softmax over a 1D numpy array
def softmax(arr):
    m = np.max(arr)
    exp_vals = np.exp(arr - m)
    return exp_vals / np.sum(exp_vals)

In [33]:
# runs online mirror descent to find approximate nash equilibrium
def equilibrium_search(
    gen_init, disc_init,
    candidates,
    T = 20,
    eta_G = 0.1, eta_D = 0.1,
    lam_G = 0.1, lam_D = 0.1
):
    gen = {
        "correct": dict(gen_init["correct"]),
        "incorrect": dict(gen_init["incorrect"])
    }
    disc = {y: dict(disc_init[y]) for y in candidates}

    Qg = {
        "correct":   {y: 0.0 for y in candidates},
        "incorrect": {y: 0.0 for y in candidates}
    }
    Qd = {y: {"correct": 0.0, "incorrect": 0.0} for y in candidates}

    for t in range(1, T + 1):
        # 1) update Q accumulators
        for v in ["correct", "incorrect"]:
            for y in candidates:
                Qg[v][y] += (1.0 / (2.0 * t)) * disc[y][v]

        for y in candidates:
            for v in ["correct", "incorrect"]:
                Qd[y][v] += (1.0 / (2.0 * t)) * gen[v][y]

        # 2) update generator policy via mirror descent
        for v in ["correct", "incorrect"]:
            logits = [
                (Qg[v][y] + lam_G * math.log(gen_init[v][y] + 1e-12)) / (1 / eta_G + lam_G)
                for y in candidates
            ]
            new_probs = softmax(np.array(logits))
            for i, y in enumerate(candidates):
                gen[v][y] = new_probs[i]

        # 3) update discriminator policy via mirror descent
        for v in ["correct", "incorrect"]:
            logits = [
                (Qd[y][v] + lam_D * math.log(disc_init[y][v] + 1e-12)) / (1 / eta_D + lam_D)
                for y in candidates
            ]
            new_probs = softmax(np.array(logits))
            for i, y in enumerate(candidates):
                disc[y][v] = new_probs[i]

    return gen, disc

## Load Model

In [35]:
def load_model(model_name):
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype = torch.float16,
        load_in_8bit = False,
        low_cpu_mem_usage = True,
        device_map = "cuda",
        trust_remote_code = True
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    return model, tokenizer

In [36]:
# mirrors subcategory_df_function from original paper
# for each row:
    # computes initial discriminator policy
    # computes initial generator policy
    # runs the equilibrium search
    # picks the final answers

def botdetect_policy_function(model, tokenizer, df):
    category_df = df.copy()

    gen_answer = []
    disc_answer = []
    gen_init_answer = []
    disc_init_answer = []
    disc_init_policy = []
    gen_init_policy = []
    disc_final_policy_consensus = []
    gen_final_policy_consensus = []

    for _, row in tqdm(category_df.iterrows(), total=len(category_df)):

        # --- discriminator init ---
        disc_init = get_initial_discriminator_probs(row, model, tokenizer)
        disc_init_policy.append(disc_init)
        gc.collect()
        torch.cuda.empty_cache()

        # --- generator init ---
        gen_init = get_initial_generator_probs(row, model, tokenizer)
        gen_init_policy.append(gen_init)
        gc.collect()
        torch.cuda.empty_cache()

        # --- initial answers (pre-equilibrium) ---
        gen_init_answer.append(max(gen_init["correct"], key=gen_init["correct"].get))
        disc_init_answer.append(max(disc_init, key=lambda choice: disc_init[choice]["correct"]))

        # --- candidates: binary ['A', 'B'] ---
        candidates = [chr(65 + i) for i in range(len(row["choices"]))]

        # --- equilibrium search (identical to original) ---
        gen_final, disc_final = equilibrium_search(
            gen_init, disc_init, candidates,
            T = 20, eta_G = 0.1, eta_D = 0.1, lam_G = 0.1, lam_D = 0.1
        )
        disc_final_policy_consensus.append(disc_final)
        gen_final_policy_consensus.append(gen_final)

        # --- Final answers (post-equilibrium) ---
        best_answer_g = pick_answer(gen_final, disc_final, candidates, method="generator")
        best_answer_d = pick_answer(gen_final, disc_final, candidates, method="discriminator")
        gen_answer.append(best_answer_g)
        disc_answer.append(best_answer_d)

    # --- Assemble output DataFrame ---
    category_df["gen_init_answer"] = gen_init_answer
    category_df["disc_init_answer"] = disc_init_answer
    category_df["gen_answer"] = gen_answer
    category_df["disc_answer"] = disc_answer
    category_df["disc_init_policy"] = disc_init_policy
    category_df["gen_init_policy"] = gen_init_policy
    category_df["disc_final_policy_consensus"] = disc_final_policy_consensus
    category_df["gen_final_policy_consensus"] = gen_final_policy_consensus

    return category_df

## Sanity Check Before Running Initial Policies

In [ ]:
# # ── Load random model and run on 50 rows for sanity check ─────────────────────────────────────────
# if 'model_d' in globals():
#     model_d.cpu()
#     del model_d
# if 'tokenizer_d' in globals():
#     del tokenizer_d

# gc.collect()
# torch.cuda.empty_cache()
# torch.cuda.synchronize()

# model_d, tokenizer_d = load_model("google/gemma-7b-it")
# df_test = df.sample(n = 50, random_state = 213).reset_index(drop=True)
# temp_test = botdetect_policy_function(model_d, tokenizer_d, df_test)

# print("True label distribution in sample:")
# print(df_test['label'].value_counts())
# print("\ndisc_init_answer distribution:")
# print(temp_test['disc_init_answer'].value_counts())

In [ ]:
# print("\nAccuracy vs true label:")
# acc = (temp_test['disc_init_answer'] == temp_test['answerKey']).mean()
# print(f"  {acc:.2%}")
# print("\n✓ If distribution is not degenerate and accuracy > 50%, proceed to full run.")

## Run Initial Policies

In [ ]:
# ── Model 1: Llama-3.1-8B-Instruct ────────────────────────────────────────────
if 'model_d' in globals():
    model_d.cpu()
    del model_d
if 'tokenizer_d' in globals():
    del tokenizer_d

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

model_d, tokenizer_d = load_model("meta-llama/Llama-3.1-8B-Instruct")
temp_df_llama = botdetect_policy_function(model_d, tokenizer_d, df)

file_path = 'InitialPolicyData/botdetect_policy_df_Llama3_8b.csv'
temp_df_llama.to_csv(file_path, index=False)
print(f"Saved: {file_path}")

In [ ]:
# ── Model 2: Mistral-7B-Instruct-v0.2 ─────────────────────────────────────────
if 'model_d' in globals():
    model_d.cpu()
    del model_d
if 'tokenizer_d' in globals():
    del tokenizer_d

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

model_d, tokenizer_d = load_model("mistralai/Mistral-7B-Instruct-v0.2")
temp_df_mistral = botdetect_policy_function(model_d, tokenizer_d, df)

file_path = 'InitialPolicyData/botdetect_policy_df_Mistral_7B.csv'
temp_df_mistral.to_csv(file_path, index=False)
print(f"Saved: {file_path}")

In [ ]:
# ── Model 3: Gemma-7b-it ──────────────────────────────────────────────────────
if 'model_d' in globals():
    model_d.cpu()
    del model_d
if 'tokenizer_d' in globals():
    del tokenizer_d

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

model_d, tokenizer_d = load_model("google/gemma-7b-it")
temp_df_gemma = botdetect_policy_function(model_d, tokenizer_d, df)

file_path = 'InitialPolicyData/botdetect_policy_df_gemma_7b.csv'
temp_df_gemma.to_csv(file_path, index=False)
print(f"Saved: {file_path}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

  1%|▍                                                                            | 63/10000 [01:24<1:36:18,  1.72it/s]

In [ ]:
# ── Model 4: Yi-1.5-9B-Chat ───────────────────────────────────────────────────
if 'model_d' in globals():
    model_d.cpu()
    del model_d
if 'tokenizer_d' in globals():
    del tokenizer_d

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

model_d, tokenizer_d = load_model("01-ai/Yi-1.5-9B-Chat")
temp_df_yi = botdetect_policy_function(model_d, tokenizer_d, df)

file_path = 'InitialPolicyData/botdetect_policy_df_Yi_9B.csv'
temp_df_yi.to_csv(file_path, index=False)
print(f"Saved: {file_path}")

In [ ]:
# ── Model 5: Granite-3.3-8b-base ──────────────────────────────────────────────
if 'model_d' in globals():
    model_d.cpu()
    del model_d
if 'tokenizer_d' in globals():
    del tokenizer_d

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

model_d, tokenizer_d = load_model("ibm-granite/granite-3.3-8b-base")
temp_df_granite = botdetect_policy_function(model_d, tokenizer_d, df)

file_path = 'InitialPolicyData/botdetect_policy_df_granite_8b.csv'
temp_df_granite.to_csv(file_path, index=False)
print(f"Saved: {file_path}")

In [ ]:
# ── Model 6: Zephyr-7b-beta ───────────────────────────────────────────────────
if 'model_d' in globals():
    model_d.cpu()
    del model_d
if 'tokenizer_d' in globals():
    del tokenizer_d

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

model_d, tokenizer_d = load_model("HuggingFaceH4/zephyr-7b-beta")
temp_df_zephyr = botdetect_policy_function(model_d, tokenizer_d, df)

file_path = 'InitialPolicyData/botdetect_policy_df_zephyr_7B.csv'
temp_df_zephyr.to_csv(file_path, index=False)
print(f"Saved: {file_path}")